# Morphology-Aware Tokenization Pipeline

This notebook combines all the separate scripts from the `scripts/` directory into a single, cohesive executable environment.
Make sure your kernel is running from the `notebook_version/` directory.

## Phase 2: Morphological Rule Application (Shielding)

In [ ]:
import os
from pathlib import Path
from functools import lru_cache
from collections import defaultdict
from tqdm import tqdm


# ── Suffix definitions ─────────────────────────────────────────────────────────
ngram_suffixes = [
    "ക്കുകയായിരുന്നു", "ന്നിരിക്കുന്നു", "ണ്ടിരിക്കുന്നു",
    "ിലായിരിക്കുന്നു", "ിലായിട്ട്", "ിട്ടുണ്ട്", "ിട്ടില്ല",
    "ുമായിരുന്നു", "ായിരുന്നു", "ച്ചിരുന്നു",
    "ന്നപ്പോൾ",
    "ിരുന്ന്", "യിരുന്നു", "ത്തിനെ",
    "ിൽനിന്ന്",
    "ിലേക്ക്", "ിലേക്കു",
    "ത്തിനായി", "ക്കായി",
    "ക്കൊണ്ട്", "ക്കൊണ്ടു", "തിനുള്ള",  "ക്കളുടെ", "യിലേക്കും", "ങ്ങളുടെ", "യിരിക്കുന്നു",  "ത്തിലാണ്" , "യുമായി" ,"ക്കാനാവില്ലെന്ന്",
    "ക്കാനാവില്ല", "ക്കാനാവും", "ക്കാനാകൂ", "ക്കാനാകില്ല",
    "ക്കാനാകാം", "ക്കാനാകുമോ", "ക്കാനാകില്ലെന്ന്", "ക്കാനാവുന്ന", "ക്കാനാവില്ലാത്ത", "ക്കാനാവില്ലാണ്", "ക്കാനാവില്ലായിരിക്കും", "ക്കാനാവില്ലായിരുന്ന",
    "ക്കാനാവില്ലായിരുന്ന്", "ക്കാനാവില്ലായിരുന്നുവോ", "ക്കാനാവില്ലായിരുന്നുവോ","ക്കുകയാണ്", "ക്കായിരിക്കുന്നു", "ക്കായിരുന്ന", "ക്കായിരുന്ന്", "ക്കായിരുന്നുവോ", "ക്കായിരുന്നുവോ", "ക്കായിരുന്നുവോ",
    "ക്കുള്ളിൽ", 
]

case_suffixes = [
    "ുടെ", "യുടെ", "ന്റെ",
    "യിൽ", "ിൽ", "ൽ",
    "ക്ക്", "ക്കു", "ിക്ക്",
    "നെ", "യെ",
    "ാൽ", "ക്കൊണ്ട്",
    "പ്പം", "ോട്",
    "വരെ", "മുതൽ",
    "ക്കൂടെ", "പ്പോലെ", "നിന്ന്", "കളിൽ", "ന്നിട്ട്", "ന്നാൽ"
]

plural_suffixes = [
    "ങ്ങൾ", "ങ്ങള്", "ക്കൾ", "ങ്ങളെ",
    "മാർ",
    "ക്കുകളുടെ", "ങ്ങളിൽ", "ക്കൾക്", "ങ്ങളിലെ"
]

verb_suffixes = [
    "ുന്നു", "ിക്കു",
    "ാണ്", "ായി", "യോടും" "ക്കും", "രിക്കുന്നത്", "രിക്കുന്ന", "രിക്കുന്നുവോ", "രിക്കുന്നുവോ", "രുന്ന്", "യിരുന്നു", "ത്തിനെ",
]

negation_suffixes = [
    "ില്ല", "ാകില്ല", "ാക്കില്ല", "ാതെ", "ാത്ത", "ാകില്ല", "ാക്കല്ല",
    "ട്ടില്ല", "ിട്ടില്ല",

]

adj_suffixes = [
    "മുള്ള", "യുള്ള",
    "പ്പെട്ട", "ക്കുക"
]
custom_suffix = [  ] 

suffixes = sorted(set(
    ngram_suffixes + case_suffixes + plural_suffixes +
    verb_suffixes + negation_suffixes + adj_suffixes + custom_suffix
), key=len, reverse=True)

SUFFIX_BY_LAST: dict[str, list[str]] = defaultdict(list)
for suf in suffixes:
    SUFFIX_BY_LAST[suf[-1]].append(suf)

WB = "▁"

# ── Path Configuration ─────────────────────────────────────────────────────────
BASE_DIR = Path.cwd().parent
INPUT_DIR = BASE_DIR / "data" / "raw_corpus"
OUTPUT_DIR = BASE_DIR / "data" / "hybrid_corpus"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


@lru_cache(maxsize=1 << 17)
def shield_word(word: str, min_stem_len: int = 3) -> str:
    stem = word
    matched_suffixes: list[str] = []

    while stem:
        matched = False
        for suf in SUFFIX_BY_LAST.get(stem[-1], []):
            if stem.endswith(suf) and len(stem) - len(suf) >= min_stem_len:
                matched_suffixes.append(suf)
                stem = stem[:-len(suf)]
                matched = True
                break
        if not matched:
            break

    if not matched_suffixes:
        return word

    return stem + "".join(WB + suf for suf in reversed(matched_suffixes))


def process_file(file_path: Path, file_bar: tqdm) -> str:
    text = file_path.read_text(encoding="utf-8")
    all_lines = [line.strip() for line in text.splitlines() if line.strip()]
    total_words = sum(len(line.split()) for line in all_lines)

    output_lines: list[str] = []
    with tqdm(
        total=total_words,
        desc=f"  ↳ {file_path.name[:30]}",
        unit="word",
        unit_scale=True,
        leave=False,
        colour="cyan",
        dynamic_ncols=True,
    ) as word_bar:
        for line in all_lines:
            words = line.split()
            output_lines.append(" ".join(shield_word(w) for w in words))
            word_bar.update(len(words))

    output_file = OUTPUT_DIR / f"hybrid_{file_path.name}"
    output_file.write_text("\n".join(output_lines) + "\n", encoding="utf-8")

    file_bar.update(1)
    cache = shield_word.cache_info()
    total = cache.hits + cache.misses
    hit_rate = (cache.hits / total * 100) if total else 0
    return f"✅ {file_path.name} ({total_words:,} words | cache hit {hit_rate:.1f}%)"


if __name__ == "__main__":
    txt_files = [
        f for f in INPUT_DIR.glob("*.txt")
        if f.is_file() and not f.name.startswith("hybrid_")
    ]

    if not txt_files:
        print(f"❌ No input .txt files found in {INPUT_DIR}")
        raise SystemExit(1)

    print(f"✅ Found {len(txt_files)} file(s) in {INPUT_DIR}\n")

    with tqdm(
        total=len(txt_files),
        desc="Overall",
        unit="file",
        colour="green",
        dynamic_ncols=True,
    ) as file_bar:
        for fp in txt_files:
            tqdm.write(process_file(fp, file_bar))

    final = shield_word.cache_info()
    print(
        f"\n🔍 Final cache — hits: {final.hits:,} | misses: {final.misses:,} | size: {final.currsize:,}"
    )
    print("🎉 Done!")


## Phase 3a: Standard BPE Training

In [ ]:
import os
from pathlib import Path
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from tokenizers.processors import TemplateProcessing

# ── Path Configuration ─────────────────────────────────────────────────────────
BASE_DIR = Path.cwd().parent
DATA_DIRECTORY = BASE_DIR / "data" / "raw_corpus"
OUTPUT_DIR = BASE_DIR / "models" / "tokenizers"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Automatically find all .txt files in the raw corpus directory
# We convert the paths to strings because the Tokenizer expects a list of string paths
corpus_files = [str(file) for file in DATA_DIRECTORY.glob("*.txt")]

# Optional: Stop the script if no files are found so it doesn't crash the trainer
if not corpus_files:
    print(f"❌ Error: No .txt files found in {DATA_DIRECTORY.resolve()}")
    exit()

print(f"✅ Found {len(corpus_files)} files for training.")

# 2. Initialize and configure the Tokenizer
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFC(),
    normalizers.Strip()
])

tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()

trainer = trainers.BpeTrainer(
    vocab_size=16000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]
)

# 3. Train using the dynamically generated list of files
tokenizer.train(files=corpus_files, trainer=trainer)

# 4. Post-Processing
tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    pair="[BOS] $A [EOS] $B:1 [EOS]:1",
    special_tokens=[
        ("[BOS]", tokenizer.token_to_id("[BOS]")),
        ("[EOS]", tokenizer.token_to_id("[EOS]"))
    ]
)

# 5. Save the model
output_path = OUTPUT_DIR / "malayalam_standard_bpe.json"
tokenizer.save(str(output_path))
print(f"✅ Tokenizer training complete and saved as {output_path}")


## Phase 3b: Hybrid BPE Training

In [ ]:
import os
from pathlib import Path
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from tokenizers.processors import TemplateProcessing

# ── Path Configuration ─────────────────────────────────────────────────────────
BASE_DIR = Path.cwd().parent
DATA_DIRECTORY = BASE_DIR / "data" / "hybrid_corpus"
OUTPUT_DIR = BASE_DIR / "models" / "tokenizers"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Automatically find all .txt files in the hybrid corpus directory
# We convert the paths to strings because the Tokenizer expects a list of string paths
corpus_files = [str(file) for file in DATA_DIRECTORY.glob("*.txt")]

# Optional: Stop the script if no files are found so it doesn't crash the trainer
if not corpus_files:
    print(f"❌ Error: No .txt files found in {DATA_DIRECTORY.resolve()}")
    exit()

print(f"✅ Found {len(corpus_files)} files for training.")

# 2. Initialize and configure the Tokenizer
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFC(),
    normalizers.Strip()
])

tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()

trainer = trainers.BpeTrainer(
    vocab_size=16000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]
)

# 3. Train using the dynamically generated list of files
tokenizer.train(files=corpus_files, trainer=trainer)

# 4. Post-Processing
tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    pair="[BOS] $A [EOS] $B:1 [EOS]:1",
    special_tokens=[
        ("[BOS]", tokenizer.token_to_id("[BOS]")),
        ("[EOS]", tokenizer.token_to_id("[EOS]"))
    ]
)

# 5. Save the model
output_path = OUTPUT_DIR / "malayalam_hybrid_bpe.json"
tokenizer.save(str(output_path))
print(f"✅ Tokenizer training complete and saved as {output_path}")


## Phase 4: Evaluation Set Generation

In [ ]:
import re
import sys
import unicodedata
from pathlib import Path

# ── Path Configuration ─────────────────────────────────────────────────────────
BASE_DIR = Path.cwd().parent

# Enable importing shield_word from 01_preprocessor.py (same directory)
# shield_word is already defined in the runtime environment from the preprocessor cell

INPUT_FILE = BASE_DIR / "data" / "raw_corpus" / "subset_8.txt"  # CRITICAL: ONLY read from the held-out test file
RAW_OUTPUT = BASE_DIR / "data" / "evaluation" / "eval_unseen_raw.txt"
GOLD_OUTPUT = BASE_DIR / "data" / "evaluation" / "eval_unseen_gold.txt"
TARGET_WORDS = 20000
MIN_LEN = 7

MALAYALAM_START = 0x0D00
MALAYALAM_END = 0x0D7F
WORD_SPLIT = re.compile(r"\s+")

def is_clean_malayalam_word(token):
    token = token.strip()
    if not token or len(token) < MIN_LEN:
        return False
    for ch in token:
        cp = ord(ch)
        if not (MALAYALAM_START <= cp <= MALAYALAM_END):
            return False
        cat = unicodedata.category(ch)
        if not cat.startswith(("L", "M")):
            return False
    return True

def main():
    words = set()
    
    if not INPUT_FILE.exists():
        print(f"Missing test file: {INPUT_FILE}")
        return

    # 1. Extract 20,000 unique unseen words
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            for token in WORD_SPLIT.split(line.strip()):
                if is_clean_malayalam_word(token):
                    words.add(token)
                    if len(words) >= TARGET_WORDS:
                        break
            if len(words) >= TARGET_WORDS:
                break

    # Ensure output directory exists
    RAW_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

    # 2. Write BOTH aligned files simultaneously
    with open(RAW_OUTPUT, "w", encoding="utf-8") as raw_f, \
         open(GOLD_OUTPUT, "w", encoding="utf-8") as gold_f:
        
        for word in sorted(words):
            raw_f.write(word + "\n")
            
            # Generate the gold standard morphological split dynamically
            gold_word = shield_word(word) 
            gold_f.write(gold_word + "\n")

    print(f"✅ Saved {len(words)} aligned words to {RAW_OUTPUT} and {GOLD_OUTPUT}")

if __name__ == "__main__":
    main()


## Phase 5a: Intrinsic Evaluation

In [ ]:
import csv
import json
import math
from collections import Counter
from pathlib import Path

from tokenizers import Tokenizer

# ── Path Configuration ─────────────────────────────────────────────────────────
BASE_DIR = Path.cwd().parent

STANDARD_TOKENIZER = str(BASE_DIR / "models" / "tokenizers" / "malayalam_standard_bpe.json")
HYBRID_TOKENIZER = str(BASE_DIR / "models" / "tokenizers" / "malayalam_hybrid_bpe.json")
RAW_WORDS_FILE = str(BASE_DIR / "data" / "evaluation" / "eval_unseen_raw.txt")
GOLD_WORDS_FILE = str(BASE_DIR / "data" / "evaluation" / "eval_unseen_gold.txt")
OUTPUT_DIR = str(BASE_DIR / "results")

def load_words(path):
    text = Path(path).read_text(encoding="utf-8")
    return [line.strip() for line in text.splitlines() if line.strip()]

def strip_markers(tok):
    for prefix in ("##", "Ġ", "▁", " "):
        if tok.startswith(prefix):
            tok = tok[len(prefix):]
    return tok.replace("</w>", "")

def percentile(values, p):
    if not values:
        return 0.0
    values = sorted(values)
    if len(values) == 1:
        return float(values[0])
    k = (len(values) - 1) * (p / 100.0)
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return float(values[int(k)])
    return float(values[f] * (c - k) + values[c] * (k - f))

def get_gold_boundaries(marked_word, marker="▁"):
    """Extracts true morphological boundary character indices from the gold word."""
    boundaries = set()
    current_len = 0
    parts = marked_word.split(marker)
    for i in range(len(parts) - 1):
        current_len += len(parts[i])
        boundaries.add(current_len)
    return boundaries

def get_pred_boundaries(visible_tokens):
    """Extracts predicted boundary character indices from tokenizer output."""
    boundaries = set()
    current_len = 0
    for i in range(len(visible_tokens) - 1):
        current_len += len(visible_tokens[i])
        boundaries.add(current_len)
    return boundaries

def evaluate(tokenizer_file, raw_words, gold_words, label):
    tokenizer = Tokenizer.from_file(tokenizer_file)
    word_rows = []
    pieces_per_word = []
    token_lengths = []
    split_count = 0
    severe_split_count = 0
    total_characters = 0
    total_tp = 0
    total_fp = 0
    total_fn = 0

    for raw_word, gold_word in zip(raw_words, gold_words):
        enc = tokenizer.encode(raw_word, add_special_tokens=False)
        tokens = enc.tokens
        visible_tokens = [strip_markers(t) for t in tokens if strip_markers(t)]
        lengths = [len(t) for t in visible_tokens]
        total_characters += sum(lengths)
        k = len(tokens)
        pieces_per_word.append(k)
        token_lengths.extend(lengths)
        if k > 1:
            split_count += 1
        if k >= 3:
            severe_split_count += 1
        gold_bounds = get_gold_boundaries(gold_word)
        pred_bounds = get_pred_boundaries(visible_tokens)
        tp = len(gold_bounds.intersection(pred_bounds))
        fp = len(pred_bounds - gold_bounds)
        fn = len(gold_bounds - pred_bounds)
        total_tp += tp
        total_fp += fp
        total_fn += fn
        word_rows.append({
            "tokenizer": label, "raw_word": raw_word, "gold_word": gold_word,
            "num_tokens": k, "tokens": " ".join(tokens),
            "visible_tokens": " ".join(visible_tokens),
            "split": 1 if k > 1 else 0, "severe_split": 1 if k >= 3 else 0,
            "true_positives": tp, "false_positives": fp, "false_negatives": fn
        })

    n = len(raw_words)
    total_tokens_generated = sum(pieces_per_word)
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    hist = dict(sorted(Counter(token_lengths).items()))
    summary = {
        "tokenizer": label, "n_words": n,
        "word_fragmentation_rate": round(split_count / n, 6) if n else 0.0,
        "severe_fragmentation_rate": round(severe_split_count / n, 6) if n else 0.0,
        "average_token_fertility": round(total_tokens_generated / n, 6) if n else 0.0,
        "characters_per_token_cpt": round(total_characters / total_tokens_generated, 6) if total_tokens_generated else 0.0,
        "morph_boundary_precision": round(precision, 6),
        "morph_boundary_recall": round(recall, 6),
        "morph_boundary_f1": round(f1_score, 6),
        "token_length_mean": round(sum(token_lengths) / len(token_lengths), 6) if token_lengths else 0.0,
        "token_length_p50": round(percentile(token_lengths, 50), 6),
        "token_length_p95": round(percentile(token_lengths, 95), 6),
        "token_length_p99": round(percentile(token_lengths, 99), 6),
        "token_length_share_len1": round(sum(x == 1 for x in token_lengths) / len(token_lengths), 6) if token_lengths else 0.0,
        "token_length_hist": json.dumps(hist, ensure_ascii=False),
    }
    return summary, word_rows

def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

def write_transposed_csv(path, std_summary, hyb_summary, delta):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys = [k for k in std_summary.keys() if k != "tokenizer"]
    rows = []
    for key in keys:
        rows.append({
            "metric": key,
            "standard_bpe": std_summary.get(key, ""),
            "hybrid_bpe": hyb_summary.get(key, ""),
            "delta_hybrid_minus_standard": delta.get(key, "")
        })
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["metric", "standard_bpe", "hybrid_bpe", "delta_hybrid_minus_standard"])
        writer.writeheader()
        writer.writerows(rows)

def main():
    raw_words = load_words(RAW_WORDS_FILE)
    gold_words = load_words(GOLD_WORDS_FILE)
    if len(raw_words) != len(gold_words):
        raise ValueError(f"CRITICAL ERROR: Data Misalignment. Raw={len(raw_words)}, Gold={len(gold_words)}")
    std_summary, std_rows = evaluate(STANDARD_TOKENIZER, raw_words, gold_words, "standard_bpe")
    hyb_summary, hyb_rows = evaluate(HYBRID_TOKENIZER, raw_words, gold_words, "hybrid_bpe")
    delta = {"tokenizer": "delta_hybrid_minus_standard"}
    for key in std_summary:
        if key == "tokenizer":
            continue
        a, b = std_summary[key], hyb_summary[key]
        delta[key] = str(round(b - a, 6)) if isinstance(a, (int, float)) and isinstance(b, (int, float)) else ""
    outdir = Path(OUTPUT_DIR)
    write_transposed_csv(outdir / "metrics_summary.csv", std_summary, hyb_summary, delta)
    write_csv(outdir / "standard_word_level.csv", std_rows)
    write_csv(outdir / "hybrid_word_level.csv", hyb_rows)
    print("✅ Evaluation Complete")
    print(f"Words evaluated: {len(raw_words)}")
    print(f"Saved (Transposed): {outdir / 'metrics_summary.csv'}")
    print(f"Saved: {outdir / 'standard_word_level.csv'}")
    print(f"Saved: {outdir / 'hybrid_word_level.csv'}")

if __name__ == "__main__":
    main()


## Phase 5b: Statistical Analysis

In [ ]:
import pandas as pd
from pathlib import Path
from scipy import stats

# ── Path Configuration ─────────────────────────────────────────────────────────
BASE_DIR = Path.cwd().parent
RESULTS_DIR = BASE_DIR / "results"

def run_statistical_analysis(std_csv_path, hyb_csv_path):
    # 1. Load the word-level data
    df_std = pd.read_csv(std_csv_path)
    df_hyb = pd.read_csv(hyb_csv_path)

    # Sanity Check: Ensure the data is perfectly paired
    if not df_std['raw_word'].equals(df_hyb['raw_word']):
        raise ValueError("Data misalignment! The raw words in both CSVs must match exactly.")

    print("=== Task 3: Comparative Statistical Analysis ===")
    
    # 2. Test 1: Token Fertility (num_tokens)
    # Are we statistically reducing the number of pieces words are broken into?
    stat_fert, p_fert = stats.wilcoxon(df_std['num_tokens'], df_hyb['num_tokens'])
    
    std_fert_mean = df_std['num_tokens'].mean()
    hyb_fert_mean = df_hyb['num_tokens'].mean()
    
    print(f"\n1. Token Fertility:")
    print(f"   Standard Mean: {std_fert_mean:.4f} | Hybrid Mean: {hyb_fert_mean:.4f}")
    print(f"   Wilcoxon p-value: {p_fert:.4e}")
    if p_fert < 0.05:
        print("   ✅ Conclusion: The difference in Token Fertility is STATISTICALLY SIGNIFICANT.")
    else:
        print("   ❌ Conclusion: The difference is NOT statistically significant.")

    # 3. Test 2: Morph-Boundary F1 Score (Approximated via True Positives/Errors)
    # We will test if the False Positives (over-segmentation) are significantly reduced.
    stat_fp, p_fp = stats.wilcoxon(df_std['false_positives'], df_hyb['false_positives'])
    
    std_fp_mean = df_std['false_positives'].mean()
    hyb_fp_mean = df_hyb['false_positives'].mean()
    
    print(f"\n2. Morph-Boundary False Positives (Over-segmentation):")
    print(f"   Standard Mean FP: {std_fp_mean:.4f} | Hybrid Mean FP: {hyb_fp_mean:.4f}")
    print(f"   Wilcoxon p-value: {p_fp:.4e}")
    if p_fp < 0.05:
        print("   ✅ Conclusion: The reduction in Boundary Violations is STATISTICALLY SIGNIFICANT.")
    else:
        print("   ❌ Conclusion: The difference is NOT statistically significant.")

if __name__ == "__main__":
    # Point these to the CSVs generated by our previous evaluation script
    STD_CSV = RESULTS_DIR / "standard_word_level.csv"
    HYB_CSV = RESULTS_DIR / "hybrid_word_level.csv"
    
    run_statistical_analysis(STD_CSV, HYB_CSV)
